[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/himanshu231204/pytorch-mastery/blob/main/05_production_engineering/experiment_tracking.ipynb)

# 03 . Experiment Tracking and Checkpointing

## Concept — Experiment Tracking & Checkpointing (revision notes)

**What it is**
- Experiment tracking = recording, for every run, the config, code version, metrics over time and artifacts, so any result can be explained and reproduced.
- Checkpointing = saving enough state (model, optimizer, scheduler, RNG, progress counters) to resume training exactly where it stopped.

**Why it matters**
- Without logs you cannot compare runs; without full-state checkpoints a crash at hour 20 costs 20 hours, and a resumed run silently differs from an uninterrupted one.
- Tools like Weights & Biases or MLflow are just nicer front-ends over the same idea: append structured records, store artifacts.

**When to use**
- Always: any run longer than a minute or any run you may want to compare later.
- Resume-capable checkpoints: spot/preemptible instances, long jobs, shared clusters.
- Best-k checkpoints: when the final epoch is not the best (overfitting) and disk is limited.

**Math & intuition**
- Exact resume requires: model weights, optimizer state (Adam moments!), LR scheduler state, epoch/step counter, and RNG states (torch CPU/CUDA, numpy, python) plus the data order.
- Atomic writes: save to `file.tmp` then `os.replace` (atomic on POSIX), so a crash mid-save never corrupts the last good checkpoint.
- JSON-lines: one JSON object per line; appendable, crash-tolerant (only the last line can be truncated), trivially loaded by pandas.

### 1. A tiny JSON-lines logger

**What this cell does (plain language):** we implement our own experiment logger in ~20 lines: each `log()` call appends one JSON object (with step and wall-clock time) to a `metrics.jsonl` file and flushes. The file format is the same idea that most tracking tools use under the hood.

In [1]:
import os, json, time, tempfile, shutil, random, copy
import numpy as np, torch, torch.nn as nn

class JsonlLogger:
    def __init__(self, run_dir):
        os.makedirs(run_dir, exist_ok=True)
        self.path = os.path.join(run_dir, "metrics.jsonl")
        self.t0 = time.time()
    def log(self, step, **metrics):
        rec = {"step": step, "time": round(time.time() - self.t0, 4), **metrics}
        with open(self.path, "a") as f:            # append mode: cheap and crash-tolerant
            f.write(json.dumps(rec) + "\n")
    def read(self):
        with open(self.path) as f:
            return [json.loads(line) for line in f if line.strip()]

root = tempfile.mkdtemp(prefix="tracking_")
lg = JsonlLogger(os.path.join(root, "demo_run"))
lg.log(0, loss=1.5, lr=0.1); lg.log(1, loss=1.2, lr=0.1)
print(open(lg.path).read())
print(lg.read())

{"step": 0, "time": 0.0, "loss": 1.5, "lr": 0.1}
{"step": 1, "time": 0.0002, "loss": 1.2, "lr": 0.1}

[{'step': 0, 'time': 0.0, 'loss': 1.5, 'lr': 0.1}, {'step': 1, 'time': 0.0002, 'loss': 1.2, 'lr': 0.1}]


### 2. Recording config, environment and code version with the run

**What this cell does (plain language):** a metric curve is useless if you cannot tell which hyperparameters and library versions produced it. We write a `config.json` snapshot with hyperparameters, Python/torch versions and (if available) the git commit hash. Run directories named by a timestamp+short hash keep runs from overwriting each other.

In [2]:
import sys, platform, subprocess, hashlib

def git_hash():
    try:
        return subprocess.run(["git", "rev-parse", "--short", "HEAD"], capture_output=True,
                              text=True, timeout=5).stdout.strip() or None
    except Exception:
        return None                                   # not a git repo / git missing

def start_run(root, config):
    tag = hashlib.md5(json.dumps(config, sort_keys=True).encode()).hexdigest()[:6]
    run_dir = os.path.join(root, f"run_{tag}")        # deterministic name from config (timestamps also common)
    os.makedirs(run_dir, exist_ok=True)
    meta = {"config": config, "torch": torch.__version__, "python": platform.python_version(),
            "platform": platform.system(), "git": git_hash()}
    with open(os.path.join(run_dir, "config.json"), "w") as f: json.dump(meta, f, indent=2)
    return run_dir

cfg = {"lr": 0.05, "epochs": 6, "hidden": 32, "dropout": 0.2, "seed": 0}
run_dir = start_run(root, cfg)
print(open(os.path.join(run_dir, "config.json")).read())

{
  "config": {
    "lr": 0.05,
    "epochs": 6,
    "hidden": 32,
    "dropout": 0.2,
    "seed": 0
  },
  "torch": "2.14.1+cu130",
  "python": "3.11.15",
  "platform": "Linux",
  "git": "72e0c54"
}


### 3. A training function with a logger (synthetic classification)

**What this cell does (plain language):** we define a tiny, fully deterministic training harness: a synthetic 2-class dataset, an MLP with dropout (so RNG state matters), AdamW and a StepLR scheduler. `train()` can start fresh or continue from an in-memory state, and logs loss/accuracy/lr each epoch to our JSON-lines logger.

In [3]:
def make_data(n=400):
    g = torch.Generator().manual_seed(7)
    X = torch.randn(n, 10, generator=g)
    y = (X[:, :3].sum(1) + 0.3 * torch.randn(n, generator=g) > 0).long()
    return X[:300], y[:300], X[300:], y[300:]
Xtr, ytr, Xva, yva = make_data()

def build(cfg):
    torch.manual_seed(cfg["seed"])
    model = nn.Sequential(nn.Linear(10, cfg["hidden"]), nn.ReLU(), nn.Dropout(cfg["dropout"]),
                          nn.Linear(cfg["hidden"], 2))
    opt = torch.optim.AdamW(model.parameters(), lr=cfg["lr"])
    sched = torch.optim.lr_scheduler.StepLR(opt, step_size=2, gamma=0.5)
    return model, opt, sched

def run_epoch(model, opt, sched):
    model.train()
    perm = torch.randperm(len(Xtr))                   # uses the GLOBAL torch RNG
    for i in range(0, len(perm), 50):
        idx = perm[i:i + 50]
        opt.zero_grad()
        nn.functional.cross_entropy(model(Xtr[idx]), ytr[idx]).backward()
        opt.step()
    sched.step()
    model.eval()
    with torch.no_grad():
        logits = model(Xva)
        return nn.functional.cross_entropy(logits, yva).item(), (logits.argmax(1) == yva).float().mean().item()

model, opt, sched = build(cfg)
logger = JsonlLogger(run_dir)
for epoch in range(cfg["epochs"]):
    vl, va = run_epoch(model, opt, sched)
    logger.log(epoch, val_loss=round(vl, 4), val_acc=round(va, 4), lr=opt.param_groups[0]["lr"])
for rec in logger.read(): print(rec)

{'step': 0, 'time': 0.0316, 'val_loss': 0.2064, 'val_acc': 0.91, 'lr': 0.05}
{'step': 1, 'time': 0.0372, 'val_loss': 0.197, 'val_acc': 0.92, 'lr': 0.025}
{'step': 2, 'time': 0.0418, 'val_loss': 0.1671, 'val_acc': 0.94, 'lr': 0.025}
{'step': 3, 'time': 0.0459, 'val_loss': 0.1688, 'val_acc': 0.93, 'lr': 0.0125}
{'step': 4, 'time': 0.0507, 'val_loss': 0.173, 'val_acc': 0.91, 'lr': 0.0125}
{'step': 5, 'time': 0.0552, 'val_loss': 0.1779, 'val_acc': 0.92, 'lr': 0.00625}


### 4. Reading logs back and finding the best epoch

**What this cell does (plain language):** because logs are plain JSON-lines, analysis needs no special tooling: pandas reads them in one line. We find the epoch with the lowest validation loss, which is the checkpoint we would want to keep.

In [4]:
import pandas as pd
df = pd.read_json(logger.path, lines=True)
print(df[["step", "val_loss", "val_acc", "lr"]].to_string(index=False))
best = df.loc[df.val_loss.idxmin()]
print("best epoch:", int(best.step), "val_loss:", best.val_loss)

 step  val_loss  val_acc      lr
    0    0.2064     0.91 0.05000
    1    0.1970     0.92 0.02500
    2    0.1671     0.94 0.02500
    3    0.1688     0.93 0.01250
    4    0.1730     0.91 0.01250
    5    0.1779     0.92 0.00625
best epoch: 2 val_loss: 0.1671


### 5. What belongs in a full checkpoint (and atomic saving)

**What this cell does (plain language):** we write `save_checkpoint` / `load_checkpoint` helpers. The checkpoint stores model, optimizer, scheduler, epoch, config AND the RNG states of torch, numpy and python. The file is written to a temporary name first and moved into place with `os.replace`, so an interrupted save never destroys the previous checkpoint.

In [5]:
def rng_state():
    return {"torch": torch.get_rng_state(),
            "cuda": torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
            "numpy": np.random.get_state(), "python": random.getstate()}

def set_rng_state(s):
    torch.set_rng_state(s["torch"])
    if s["cuda"] is not None and torch.cuda.is_available(): torch.cuda.set_rng_state_all(s["cuda"])
    np.random.set_state(s["numpy"]); random.setstate(s["python"])

def save_checkpoint(path, model, opt, sched, epoch, cfg, metric=None):
    state = {"model": model.state_dict(), "optimizer": opt.state_dict(), "scheduler": sched.state_dict(),
             "epoch": epoch, "config": cfg, "metric": metric, "rng": rng_state()}
    tmp = path + ".tmp"
    torch.save(state, tmp)                  # write fully...
    os.replace(tmp, path)                   # ...then atomically swap into place

def load_checkpoint(path, model, opt, sched):
    state = torch.load(path, weights_only=False)   # RNG states contain numpy objects -> not weights-only
    model.load_state_dict(state["model"]); opt.load_state_dict(state["optimizer"])
    sched.load_state_dict(state["scheduler"]); set_rng_state(state["rng"])
    return state["epoch"]

p = os.path.join(root, "ckpt.pt")
save_checkpoint(p, model, opt, sched, epoch=cfg["epochs"] - 1, cfg=cfg)
ck = torch.load(p, weights_only=False)
print("checkpoint keys:", list(ck.keys()))
print("optimizer state keys (Adam moments):", list(ck["optimizer"]["state"][0].keys()))
print("file size (bytes):", os.path.getsize(p), "| leftover .tmp:", os.path.exists(p + ".tmp"))

checkpoint keys: ['model', 'optimizer', 'scheduler', 'epoch', 'config', 'metric', 'rng']
optimizer state keys (Adam moments): ['step', 'exp_avg', 'exp_avg_sq']
file size (bytes): 23856 | leftover .tmp: False


### 6. The resume test: interrupted + resumed run == uninterrupted run

**What this cell does (plain language):** the real test of a checkpoint is exact equivalence. We train 6 epochs straight through, then train 3 epochs, save, rebuild EVERYTHING from scratch, load the checkpoint, and train 3 more. If we saved the right state, the final weights match bit for bit.

In [6]:
def train_n(model, opt, sched, n):
    for _ in range(n): run_epoch(model, opt, sched)

def flat(m): return torch.cat([p.detach().flatten() for p in m.parameters()])

# A) uninterrupted: 6 epochs
set_rng_state({"torch": torch.manual_seed(123).get_state(), "cuda": None,
               "numpy": np.random.RandomState(123).get_state(), "python": random.Random(123).getstate()})
mA, oA, sA = build(cfg); train_n(mA, oA, sA, 6)

# B) interrupted at epoch 3, then resumed from the checkpoint in a *fresh* model/optimizer
set_rng_state({"torch": torch.manual_seed(123).get_state(), "cuda": None,
               "numpy": np.random.RandomState(123).get_state(), "python": random.Random(123).getstate()})
mB, oB, sB = build(cfg); train_n(mB, oB, sB, 3)
pB = os.path.join(root, "mid.pt"); save_checkpoint(pB, mB, oB, sB, epoch=2, cfg=cfg)
del mB, oB, sB
mC, oC, sC = build(cfg)                       # brand new objects
load_checkpoint(pB, mC, oC, sC)               # restores weights, Adam moments, LR schedule, RNG
train_n(mC, oC, sC, 3)
print("exact resume (bitwise identical weights):", torch.equal(flat(mA), flat(mC)))
print("max abs diff:", (flat(mA) - flat(mC)).abs().max().item())

exact resume (bitwise identical weights): True
max abs diff: 0.0


### 7. What goes wrong if you skip pieces of the state

**What this cell does (plain language):** we repeat the resume experiment three times, each time deliberately NOT restoring one component (the optimizer, the scheduler, or the RNG) and measure how far the final weights drift from the uninterrupted run. This shows concretely why each piece is needed.

In [7]:
def resume_without(skip):
    set_rng_state({"torch": torch.manual_seed(123).get_state(), "cuda": None,
                   "numpy": np.random.RandomState(123).get_state(), "python": random.Random(123).getstate()})
    m, o, s = build(cfg); train_n(m, o, s, 3)
    ck = {"model": copy.deepcopy(m.state_dict()), "opt": copy.deepcopy(o.state_dict()),
          "sch": copy.deepcopy(s.state_dict()), "rng": rng_state()}
    m2, o2, s2 = build(cfg)
    m2.load_state_dict(ck["model"])
    if skip != "optimizer": o2.load_state_dict(ck["opt"])
    if skip != "scheduler": s2.load_state_dict(ck["sch"])
    if skip != "rng": set_rng_state(ck["rng"])
    else: torch.manual_seed(999)                    # a "fresh" seed, as a naive restart would use
    train_n(m2, o2, s2, 3)
    return (flat(mA) - flat(m2)).abs().max().item()

for skip in ("nothing", "optimizer", "scheduler", "rng"):
    print(f"skipping {skip:9s}: max weight diff vs uninterrupted run = {resume_without(skip):.3e}")

skipping nothing  : max weight diff vs uninterrupted run = 0.000e+00


skipping optimizer: max weight diff vs uninterrupted run = 4.051e-01


skipping scheduler: max weight diff vs uninterrupted run = 4.448e-02


skipping rng      : max weight diff vs uninterrupted run = 1.014e-01


### 8. Best-k checkpoint manager

**What this cell does (plain language):** we write a small class that keeps only the k best checkpoints by a metric (lower is better) and deletes the rest, plus a `last.pt` that is always overwritten for resuming. This is the same policy as `ModelCheckpoint(save_top_k=k)` in Lightning.

In [8]:
import heapq
class BestK:
    def __init__(self, dirpath, k=2):
        self.dir, self.k, self.heap = dirpath, k, []         # heap of (-metric, path): root = worst kept
        os.makedirs(dirpath, exist_ok=True)
    def update(self, metric, epoch, model, opt, sched, cfg):
        save_checkpoint(os.path.join(self.dir, "last.pt"), model, opt, sched, epoch, cfg, metric)
        path = os.path.join(self.dir, f"epoch{epoch}_loss{metric:.4f}.pt")
        if len(self.heap) < self.k:
            save_checkpoint(path, model, opt, sched, epoch, cfg, metric); heapq.heappush(self.heap, (-metric, path))
        elif metric < -self.heap[0][0]:                      # better than the worst kept one
            save_checkpoint(path, model, opt, sched, epoch, cfg, metric)
            _, old = heapq.heapreplace(self.heap, (-metric, path)); os.remove(old)
    def best_paths(self): return sorted((-m, os.path.basename(p)) for m, p in self.heap)

mgr = BestK(os.path.join(root, "ckpts"), k=2)
m, o, s = build(cfg)
for epoch in range(6):
    vl, _ = run_epoch(m, o, s)
    mgr.update(vl, epoch, m, o, s, cfg)
    print(f"epoch {epoch}: val_loss={vl:.4f}")
print("kept best-2:", mgr.best_paths())
print("files on disk:", sorted(os.listdir(mgr.dir)))

epoch 0: val_loss=0.2064


epoch 1: val_loss=0.1970


epoch 2: val_loss=0.1671


epoch 3: val_loss=0.1688


epoch 4: val_loss=0.1730


epoch 5: val_loss=0.1779
kept best-2: [(0.16712647676467896, 'epoch2_loss0.1671.pt'), (0.16884969174861908, 'epoch3_loss0.1688.pt')]
files on disk: ['epoch2_loss0.1671.pt', 'epoch3_loss0.1688.pt', 'last.pt']


### 9. Hosted trackers (wandb, mlflow): guarded

**What this cell does (plain language):** the same logging calls map onto hosted tools. We try to import them; if missing we print how to install and carry on. When installed, we run them in OFFLINE/local mode so no account or network is needed. Note how little changes: `log(step, **metrics)` becomes `wandb.log` / `mlflow.log_metrics`.

In [9]:
try:
    import wandb
    os.environ["WANDB_MODE"] = "offline"                       # no network, no login
    run = wandb.init(project="demo", dir=root, config=cfg, reinit=True)
    wandb.log({"val_loss": 0.5}, step=0); run.finish()
    print("wandb offline run logged")
except ImportError:
    print("wandb not installed -> `pip install wandb` (then wandb.init / wandb.log); skipping")
except Exception as e:
    print("wandb present but could not run offline:", type(e).__name__)

try:
    import mlflow
    mlflow.set_tracking_uri("file://" + os.path.join(root, "mlruns"))   # local folder store
    with mlflow.start_run():
        mlflow.log_params(cfg); mlflow.log_metric("val_loss", 0.5, step=0)
    print("mlflow run logged to", os.path.join(root, "mlruns"))
except ImportError:
    print("mlflow not installed -> `pip install mlflow` (then mlflow.start_run / log_metric); skipping")
except Exception as e:
    print("mlflow present but could not run:", type(e).__name__)

wandb not installed -> `pip install wandb` (then wandb.init / wandb.log); skipping
mlflow not installed -> `pip install mlflow` (then mlflow.start_run / log_metric); skipping


### 10. Cleanup

**What this cell does (plain language):** we remove the temporary directory holding all runs and checkpoints from the demos.

In [10]:
shutil.rmtree(root, ignore_errors=True)
print("removed:", not os.path.exists(root))

removed: True


## Common bugs

- **Saving only `model.state_dict()` and calling it a checkpoint.** Fix: Also save optimizer, scheduler, epoch/step, RNG states and config; otherwise a resume restarts Adam's moments and LR schedule.
- **Not restoring the scheduler, so a resumed run restarts at the initial LR.** Fix: Save and load `scheduler.state_dict()`, and verify `opt.param_groups[0]['lr']` after loading.
- **Writing the checkpoint file in place.** Fix: Write to a temp file and `os.replace`; a crash mid-write otherwise corrupts the only copy.
- **`torch.load` on a checkpoint with non-tensor objects fails (or is unsafe) with `weights_only=True`.** Fix: Only load trusted files; use `weights_only=False` when you store RNG/numpy objects, or store those separately.
- **Loading a checkpoint saved from a DDP/compiled model into a plain model gives `module.` / `_orig_mod.` key mismatches.** Fix: Save `model.module.state_dict()`, or strip the prefix when loading.
- **Selecting the 'best' checkpoint on the test set.** Fix: Pick by validation metric; touch the test set once at the end.
- **Logging metrics only at the end of training.** Fix: Log every epoch (or N steps) so crashes and divergence are visible; flush to disk each write.
- **Moving a checkpoint to a different device without `map_location`.** Fix: Use `torch.load(path, map_location='cpu')` then `.to(device)`.
- **Runs overwrite each other because they share a directory.** Fix: Make run directories unique (config hash / timestamp) and store the config alongside the metrics.

## Exercises

**Exercise 1.** Add a `log_many(records)` method to `JsonlLogger` that appends several records in one file open, and test it with 3 records.

In [11]:
# Your attempt for Exercise 1 here

**Solution 1**

In [12]:
# Solution 1
import tempfile, os, json, time
class JsonlLogger2(JsonlLogger):
    def log_many(self, records):
        with open(self.path, "a") as f:
            for step, m in records: f.write(json.dumps({"step": step, **m}) + "\n")
l2 = JsonlLogger2(tempfile.mkdtemp())
l2.log_many([(0, {"loss": 1.0}), (1, {"loss": 0.8}), (2, {"loss": 0.6})])
print(len(l2.read()))

3


**Exercise 2.** Write `latest_checkpoint(dir)` that returns the path of the file with the largest epoch number in names like `epoch3_loss0.5.pt` (or None).

In [13]:
# Your attempt for Exercise 2 here

**Solution 2**

In [14]:
# Solution 2
import re, os
def latest_checkpoint(d):
    best, best_ep = None, -1
    for f in os.listdir(d):
        m = re.match(r"epoch(\d+)_", f)
        if m and int(m.group(1)) > best_ep: best, best_ep = os.path.join(d, f), int(m.group(1))
    return best
d = tempfile.mkdtemp()
for e in (1, 5, 3): open(os.path.join(d, f"epoch{e}_loss0.5.pt"), "w").close()
print(latest_checkpoint(d))

/tmp/tmpd9r83_rk/epoch5_loss0.5.pt


**Exercise 3.** Verify that the optimizer checkpoint contains Adam's moments: after one epoch of training, print the shape and `step` of `exp_avg` for the first parameter from `opt.state_dict()`.

In [15]:
# Your attempt for Exercise 3 here

**Solution 3**

AdamW keeps exp_avg and exp_avg_sq per parameter, which is why optimizer state doubles checkpoint size.

In [16]:
# Solution 3
m, o, s = build(cfg)
for _ in range(1): run_epoch(m, o, s)
st = o.state_dict()["state"][0]
print(st["exp_avg"].shape, st["step"])

torch.Size([32, 10]) tensor(6.)


**Exercise 4.** Modify `BestK` so that higher is better (e.g. accuracy). What minimal changes are needed?

In [17]:
# Your attempt for Exercise 4 here

**Solution 4**

In [18]:
# Solution 4
# Only the comparison direction flips: store (metric, path) in the min-heap so the worst (smallest) kept metric is at the root,
# and replace when the new metric is GREATER than heap[0][0].
import heapq
heap = []; k = 2
for acc in (0.6, 0.7, 0.65, 0.9):
    if len(heap) < k: heapq.heappush(heap, (acc, f"acc{acc}"))
    elif acc > heap[0][0]: heapq.heapreplace(heap, (acc, f"acc{acc}"))
print(sorted(heap, reverse=True))

[(0.9, 'acc0.9'), (0.7, 'acc0.7')]


**Exercise 5.** Show the difference between `torch.save(model, path)` (pickles the class) and saving `state_dict()`: save both and compare file sizes; explain which is safer to refactor.

In [19]:
# Your attempt for Exercise 5 here

**Solution 5**

state_dict() stores only tensors, so code can be refactored freely; whole-model pickles break when the class moves.

In [20]:
# Solution 5
import io
b1, b2 = io.BytesIO(), io.BytesIO()
m, _, _ = build(cfg)
torch.save(m, b1); torch.save(m.state_dict(), b2)
print(len(b1.getvalue()), len(b2.getvalue()))

5277 3933
